# Code Generator and Unit Tester

A tool that takes a description of what you want to build, generates the code, and writes unit test cases for it. Powered by GPT and Llama running side by side so you can compare outputs.

In [1]:
# imports
import os
from openai import OpenAI
from dotenv import load_dotenv
import gradio as gr

In [2]:
# constants

MODEL_GPT    = 'gpt-4o-mini'
MODEL_OLLAMA = 'llama3.1:8b'

OLLAMA_BASE_URL = 'http://localhost:11434/v1'

LANGUAGES = ['Python', 'JavaScript']

In [3]:
# set up environment

load_dotenv()

openai_api_key = os.getenv('OPENAI_API_KEY')

if not openai_api_key:
    print("❌ OpenAI API key not found")
else:
    print("✅ OpenAI API key found!")

gpt_client    = OpenAI(api_key=openai_api_key)
ollama_client = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

✅ OpenAI API key found!


In [4]:
# system prompt

system_prompt = """
You are an expert software engineer. When given a description of what to build,
you generate clean, well-structured code and thorough unit tests.
Always respond with only the raw code — no explanations, no markdown fences.
"""

In [5]:
# helper functions

def build_code_prompt(description, language):
    return (
        f"Write a {language} function or script that does the following:\n"
        f"{description}\n\n"
        f"Respond with only the raw {language} code. No explanations, no markdown."
    )

def build_test_prompt(code, language):
    return (
        f"Write unit tests for the following {language} code:\n\n"
        f"{code}\n\n"
        f"Use the standard testing framework for {language}. "
        f"Respond with only the raw {language} test code. No explanations, no markdown."
    )

def call_model(client, model, prompt):
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": prompt}
        ]
    )
    return response.choices[0].message.content.strip()

def generate_all(description, language):
    lang = language.lower()

    gpt_code    = call_model(gpt_client,    MODEL_GPT,    build_code_prompt(description, language))
    ollama_code = call_model(ollama_client, MODEL_OLLAMA, build_code_prompt(description, language))

    gpt_tests    = call_model(gpt_client,    MODEL_GPT,    build_test_prompt(gpt_code, language))
    ollama_tests = call_model(ollama_client, MODEL_OLLAMA, build_test_prompt(ollama_code, language))

    return (
        gr.update(value=gpt_code,    language=lang),
        gr.update(value=gpt_tests,   language=lang),
        gr.update(value=ollama_code, language=lang),
        gr.update(value=ollama_tests,language=lang),
    )


In [6]:
# Gradio UI

with gr.Blocks() as ui:
    gr.Markdown("## 🛠️ Code Generator & Unit Test Writer")
    gr.Markdown(
        "Describe what you want to build, choose a language, and let "
        "GPT and Llama generate the code and unit tests side by side."
    )

    with gr.Row():
        description_input = gr.Textbox(
            lines=3,
            placeholder="e.g. A function that takes a list of numbers and returns the top 3 largest values",
            label="Describe what you want to build"
        )
        language_picker = gr.Radio(
            choices=LANGUAGES,
            value='Python',
            label="Language"
        )

    generate_btn = gr.Button("Generate", variant="primary")

    with gr.Row():
        with gr.Column():
            gr.Markdown(f"### 🤖 GPT ({MODEL_GPT})")
            gpt_code_output  = gr.Code(label="Generated Code",  language="python", lines=15)
            gpt_test_output  = gr.Code(label="Unit Tests",       language="python", lines=15)
        with gr.Column():
            gr.Markdown(f"### 🦙 Llama ({MODEL_OLLAMA})")
            ollama_code_output = gr.Code(label="Generated Code", language="python", lines=15)
            ollama_test_output = gr.Code(label="Unit Tests",      language="python", lines=15)

    generate_btn.click(
        fn=lambda: gr.update(interactive=False),
        outputs=generate_btn
    ).then(
        fn=generate_all,
        inputs=[description_input, language_picker],
        outputs=[gpt_code_output, gpt_test_output, ollama_code_output, ollama_test_output]
    ).then(
        fn=lambda: gr.update(interactive=True),
        outputs=generate_btn
    )

ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
